# Mission 1: Image Enhancement

ARGUS has a dark but recoverable still. Enhance it and describe one detail that is visible in the image. A timestamp may be one useful observation, but the clue is not a fixed answer and enhancement does not prove a time of death.

## Learning objectives
Inspect shape and BGR channels; convert to grayscale; compare brightness, equalization, CLAHE, blur and thresholding; crop and save the timestamp/camera overlay.

## Setup and input
From the repository root run `python -m pip install -e .` and launch Jupyter with `jupyter lab`. Input: photographic simulated CCTV practice frame at `assets/images/case_cctv_01.png`. This is a crop/tonal derivative of a credited public-domain archival laboratory photograph, not real footage from the fictional case. OpenCV reads color as BGR. This headless package cannot open desktop windows; use notebook display instead.

Small exercise: predict what `beta=90` does before running the adjustment.

In [ ]:
from pathlib import Path
import sys
import cv2
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from the_last_chai.image_enhancement import load_image, image_info, adjust_brightness_contrast, enhance_image, crop_image, save_image
image = load_image(str(ROOT / 'assets/images/case_cctv_01.png'))
image_info(image), image.shape

`imread` loads pixels into a NumPy array. `shape` is height, width, channels; three channels here are BGR, not RGB. `imshow` is intentionally avoided because it needs a desktop GUI.

In [ ]:
plt.figure(figsize=(12, 5))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off');

Brightness adds an offset (`beta`); contrast multiplies values (`alpha`). Grayscale keeps intensity only. Equalization globally spreads contrast, while CLAHE works on small tiles to limit over-amplification. Blur smooths noise but can erase detail; thresholding creates a binary image and is not always useful for text.

Exercise: compare `beta=90` with `beta=15`. Which one clips bright pixels?

In [ ]:
adjusted = adjust_brightness_contrast(image, alpha=1.25, beta=18)
results = enhance_image(image)
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for axis, (name, pixels) in zip(axes, results.items()):
    axis.imshow(pixels, cmap='gray'); axis.set_title(name); axis.axis('off')
plt.tight_layout();

In [ ]:
# TODO: Compare a gentle brightness increase with an excessive one. What detail clips?
trial = enhance_image(adjust_brightness_contrast(image, alpha=1.0, beta=70))
plt.figure(figsize=(10, 3)); plt.imshow(trial['clahe'], cmap='gray'); plt.title('Your brightness experiment'); plt.axis('off');

Crop coordinates are x/y/width/height in the image. The simulated camera overlay is at the top edge; inspect both corners because camera layouts vary. CLAHE is a readable reference, not proof that every CCTV image can be repaired. `cv2.imwrite` saves the result.

Hint 1: compare the source and enhanced frame. Hint 2: inspect enlarged crops from both top corners. Hint 3: describe one detail visible in this selected image.

Clue submission: the portal records a free-response observation about the processed image; it does not grade every image against `23:42`. The case timestamp is a fictional overlay, not verified incident time. Challenge: why can no enhancement recover a detail whose pixels were never recorded? Common errors: `imread` returns `None` when a path is wrong; Matplotlib expects RGB for color display.

In [ ]:
height, width = results['clahe'].shape[:2]
corner_width = int(width * 0.42)
corner_height = int(height * 0.18)
clock_left = crop_image(results['clahe'], 0, 0, corner_width, corner_height)
clock_right = crop_image(results['clahe'], width - corner_width, 0, corner_width, corner_height)
resized = cv2.resize(image, (360, 210))
histogram = cv2.calcHist([results['gray']], [0], None, [256], [0, 256])
print('Resized shape:', resized.shape, '| histogram bins:', histogram.shape[0], '| total pixels:', int(histogram.sum()))
save_image(str(ROOT / 'outputs/level1_enhanced.png'), results['clahe'])
save_image(str(ROOT / 'outputs/level1_clock_crop.png'), clock_left)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(clock_left, cmap='gray'); axes[0].set_title('Enlarged upper-left overlay'); axes[0].axis('off')
axes[1].imshow(clock_right, cmap='gray'); axes[1].set_title('Enlarged upper-right overlay'); axes[1].axis('off')
plt.tight_layout();

## Recap
We loaded a real image array, inspected channel order, compared transformations, cropped the evidence and saved outputs. We read a staged overlay; an enhancement algorithm did not independently determine the incident time.